# ACM-CAP scene properties


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import pickle
import h5py
import numpy as np
import pandas as pd
import xarray as xr

# Read pandas HDF5 blocks without PyTables.
def read_pandas_fixed_hdf(path):
    h5 = h5py.File(path, "r")
    group = h5["StCu"]
    blocks = []
    for key in group:
        if key.endswith("_items"):
            values = group[key.replace("_items", "_values")]
            values = pickle.loads(values[0]) if values.dtype == object else values[:]
            blocks.append(pd.DataFrame(values, index=group["axis1"][:], columns=group[key][:].astype(str)))
    df = pd.concat(blocks, axis=1)[group["axis0"][:].astype(str)]
    h5.close()
    df["time_mid"] = pd.to_datetime(df["time_mid"])
    return df

SOURCE = Path("../../resources_mesoscale_structure/ACMCAP_judge_StCu.h5")
df = read_pandas_fixed_hdf(SOURCE)
OUTPUT_ZARR = Path("output/earthcare-marine-stratocumulus-transitions/acmcap_judge_stcu/earthcode_acmcap_judge_stcu.zarr")
OUTPUT_ZARR.parent.mkdir(parents=True, exist_ok=True)


## Labels and coordinates


In [15]:
CLASS_NAMES = {
    0: 'Solid stratus',
    1: 'Closed MCC',
    2: 'Open MCC',
    3: 'Disorganized MCC',
    4: 'Clustered Cu',
    5: 'Suppressed Cu',
}

REGION_NAMES = {
    0: 'SEA',
    1: 'NEA',
    2: 'SEP',
    3: 'NEP',
}

In [16]:
df["cat_name"] = df.cats.map(CLASS_NAMES)
df["region"] = df.stratus_region_index.map(REGION_NAMES)
for name in ["scene_id", "stratus_region_index", "cats", "rain_n_clusters"]:
    df[name] = df[name].astype("int32")
df.index = pd.Index(np.arange(len(df), dtype="int32"), name="scene")
ds = df.to_xarray()
for name in df.select_dtypes("object"):
    ds[name] = ds[name].astype(str)
ds = ds.set_coords(["scene_id", "time_mid", "lat_mid", "lon_mid", "orbit", "stratus_region_index", "region"])
ds


/var/folders/k6/1r9h76z14xx_l3r9n7cdxc140000gn/T/ipykernel_69372/2347209287.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for name in df.select_dtypes("object"):


<xarray.Dataset> Size: 49MB
Dimensions:                               (scene: 51861)
Coordinates:
  * scene                                 (scene) int32 207kB 0 1 ... 51860
    scene_id                              (scene) int32 207kB 0 1 ... 51860
    stratus_region_index                  (scene) int32 207kB 0 0 0 0 ... 3 3 3
    time_mid                              (scene) datetime64[ns] 415kB 2024-0...
    lon_mid                               (scene) float64 415kB 6.947 ... -134.6
    lat_mid                               (scene) float64 415kB -3.058 ... 21.94
    orbit                                 (scene) <U6 1MB '01005E' ... '08371E'
    region                                (scene) <U3 622kB 'SEA' ... 'NEP'
Data variables: (12/103)
    along_track_mid                       (scene) float64 415kB 128.5 ... 128.5
    lat_min                               (scene) float64 415kB -3.532 ... 21.47
    lat_max                               (scene) float64 415kB -2.493 ... 22.5
    ice_flag                              (scene) float64 415kB 0.0 0.0 ... 0.0
    cwp_skew                              (scene) float64 415kB 0.3971 ... 7.588
    cwp_inhomogeneity                     (scene) float64 415kB 0.1773 ... 2.263
    ...                                    ...
    rwc_cbh_perc95                        (scene) float64 415kB nan ... 6.485...
    rwc_cbh_perc05                        (scene) float64 415kB nan nan ... 0.0
    rain_gini                             (scene) float64 415kB 0.0 ... 0.9391
    rain_lcf                              (scene) float64 415kB 0.0 ... 0.625
    rain_n_clusters                       (scene) int32 207kB 0 0 0 0 ... 1 2 2
    cat_name                              (scene) <U16 3MB 'Suppressed Cu' .....

## Source attributes


In [17]:
import sys
import fsspec
sys.path.insert(0, "..")
import access_ec_on_maap as maap

REGION_BOUNDS = {name: maap.bbox_regions[name] for name in REGION_NAMES.values()}

# Native EarthCARE MSI/ACM-CAP variables and their scene counterparts.
SOURCE_VARIABLES = {
    "MSI_COP_2A": {
        "cloud_water_path": ["cloud_cover", "cwp_inhomogeneity", "cwp_skew"],
        "latitude": ["lat_min", "lat_mid", "lat_max"],
        "longitude": ["lon_mid"],
        "time": ["time_mid"],
    },
    "MSI_NOM_1B": {"pixel_values": ["ice_flag"]},
    "ACM_CAP_2B": {
        "liquid_effective_radius": ["ER", "ER_passive_perspective"],
        "liquid_classification": ["N_cloudy_pixels", "liquid_classification_mean"],
        "liquid_number_concentration": ["Nd"],
        "liquid_number_concentration_error": ["Nd_error"],
        "height": ["cbh", "cbh_m", "cth", "cth_m", "cth_all_clouds", "multilayer"],
        "CPR_doppler_velocity": ["doppler_velocity_mean"],
        "liquid_optical_depth": ["liquid_optical_depth"],
        "liquid_water_content": ["lwp"],
        "liquid_water_content_error": ["lwp_error"],
        "rain_median_volume_diameter": ["rain_median_volume_diameter"],
        "rain_normalized_number_concentration": ["rain_normalized_number_concentration"],
        "rain_rate": ["rain_rate_cbh", "rain_rate_max", "rain_rate_mean", "rain_rate_subcloud", "rain_rate_surface"],
        "rain_water_content": ["rwc_cbh", "rwp", "rwp_subcloud"],
    },
}

# Read metadata only, using BA products from the first scene's orbit.
fs = fsspec.filesystem("https", headers={"Authorization": f"Bearer {maap.get_token()}"}, **maap.io_params["fsspec_params"])
orbit = str(ds.orbit.values[0])
for product, variables in SOURCE_VARIABLES.items():
    collection = "EarthCAREL1Validated_MAAP" if product == "MSI_NOM_1B" else "EarthCAREL2Validated_MAAP"
    url = maap.get_ec_data_url_via_maap(None, product, frame=orbit[-1], orbitNumber=int(orbit[:-1]), productVersion="ba", ec_collection=[collection], max_items=1)[0][0]
    remote = fs.open(url, "rb")
    native = xr.open_dataset(remote, engine="h5netcdf", group="ScienceData", decode_cf=False)
    for source, names in variables.items():
        attrs = {key: value for key, value in native[source].attrs.items() if key in ["long_name", "standard_name", "units", "definition", "notes", "comment"] and isinstance(value, str) and value}
        for name in names:
            for key, value in attrs.items():
                ds[name].attrs[key] = value
            ds[name].attrs["source_product"] = product
            ds[name].attrs["source_variable"] = source
    native.close()
    remote.close()


Accessing 1 items (limited by max_items).
1 items found that matched the query.
Found .h5 file in enclosure_h5: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/MSI_COP_2A/BA/2024/08/01/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E.h5
HDF5 URL: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/MSI_COP_2A/BA/2024/08/01/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E/ECA_EXBA_MSI_COP_2A_20240801T132709Z_20250904T090505Z_01005E.h5
Orbit: 01005E


/opt/miniconda3/envs/pangeo-new/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'argo' loading failed:
cannot import name 'collections_to_dsk' from 'dask.base' (/opt/miniconda3/envs/pangeo-new/lib/python3.12/site-packages/dask/base.py)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


Accessing 1 items (limited by max_items).
1 items found that matched the query.
Found .h5 file in enclosure_h5: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/MSI_NOM_1B/BA/2024/08/01/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E.h5
HDF5 URL: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/MSI_NOM_1B/BA/2024/08/01/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E/ECA_EXBA_MSI_NOM_1B_20240801T132709Z_20250620T211935Z_01005E.h5
Orbit: 01005E
Accessing 1 items (limited by max_items).
1 items found that matched the query.
Found .h5 file in enclosure_h5: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/ACM_CAP_2B/BA/2024/08/01/ECA_EXBA_ACM_CAP_2B_20240801T132709Z_20250905T073323Z_01005E/ECA_EXBA_ACM_CAP_2B_20240801T132709Z_20250905T073

## Scene metadata


In [18]:
# Shared scene definitions.
for names in SOURCE_VARIABLES["ACM_CAP_2B"].values():
    for name in names:
        ds[name].attrs["source_representation"] = "2D vertical profile"
        ds[name].attrs["intermediate_representation"] = "1D along satellite track"
        ds[name].attrs["intermediate_resolution"] = "1 km"
        ds[name].attrs["final_observation_unit"] = "128 km MSI neural-network scene"
        ds[name].attrs["scene_summary"] = "mean"
        ds[name].attrs["cell_methods"] = "scene: mean"
for name in ["cloud_cover", "cwp_inhomogeneity", "cwp_skew", "ice_flag"]:
    ds[name].attrs["source_representation"] = "2D horizontal imager variable"
    ds[name].attrs["scene_summary"] = "mean"
    ds[name].attrs["cell_methods"] = "scene: mean"
    ds[name].attrs["units"] = "1"
for name in ["rain_fraction", "rain_gini", "rain_lcf", "rain_n_clusters"]:
    ds[name].attrs["source_product"] = "ACM_CAP_2B"
    ds[name].attrs["source_variable"] = "rain_rate"
    ds[name].attrs["final_observation_unit"] = "128 km MSI neural-network scene"
    ds[name].attrs["units"] = "1"


In [19]:
for name in ["lwp", "rwp", "rwp_subcloud"]:
    ds[name].attrs["long_name"] = ds[name].attrs["long_name"].lower().replace("content", "path")
    ds[name].attrs["units"] = ds[name].attrs["units"].replace("m-3", "m-2")
ds["rwp_subcloud"].attrs["long_name"] = "sub-cloud " + ds["rwp"].attrs["long_name"]

for name, suffix in {"cwp_inhomogeneity": "inhomogeneity", "cwp_skew": "skewness"}.items():
    ds[name].attrs["long_name"] = ds[name].attrs["long_name"].lower() + " " + suffix
    
for prefix, title in {"cbh": "Cloud base height", "cth": "Cloud top height"}.items():
    for name in [prefix, prefix + "_m"]:
        ds[name].attrs["long_name"] = title
        ds[name].attrs["source_variable"] = "liquid_classification"
    ds[prefix].attrs["units"] = "JSG_height"
    
for suffix, label in {"min": "Minimum", "max": "Maximum"}.items():
    ds[f"lat_{suffix}"].attrs["long_name"] = f"{label} latitude of the scene."
    ds[f"lat_{suffix}"].attrs["description"] = f"{label} latitude of the scene along the nadir track."

ds["along_track_mid"].attrs["long_name"] = "midpoint along-track index"
ds["along_track_mid"].attrs["units"] = "1"
ds["cat_name"].attrs["long_name"] = "CNN mesoscale cloud morphology class name"
ds["cats"].attrs["long_name"] = "CNN mesoscale cloud morphology class"
ds["cats"].attrs["units"] = "1"
ds["cats"].attrs["flag_values"] = np.array(list(CLASS_NAMES), dtype="int32")
ds["cats"].attrs["flag_meanings"] = " ".join(name.lower().replace(" ", "_") for name in CLASS_NAMES.values())
ds["cert"].attrs["long_name"] = "CNN morphology classification confidence"
ds["cert"].attrs["units"] = "1"
ds["orbit"].attrs["long_name"] = "EarthCARE orbit and frame identifier"
ds["region"].attrs["long_name"] = "stratocumulus region code"
ds["region"].attrs["region_bounds"] = str(REGION_BOUNDS)
ds["scene"].attrs["long_name"] = "scene index"
ds["scene"].attrs["units"] = "1"
ds["scene_id"].attrs["long_name"] = "scene identifier"
ds["scene_id"].attrs["units"] = "1"
ds["stratus_region_index"].attrs["long_name"] = "stratocumulus region index"
ds["stratus_region_index"].attrs["units"] = "1"
ds["stratus_region_index"].attrs["flag_values"] = np.array(list(REGION_NAMES), dtype="int32")
ds["stratus_region_index"].attrs["flag_meanings"] = "southeast_atlantic northeast_atlantic southeast_pacific northeast_pacific"
ds['lat_mid'].attrs['description'] = "Latitude of the center of the scene."
ds['lon_mid'].attrs['description'] = "Longitude of the center of the scene."
ds["cloud_cover"].attrs['long_name'] = "MSI cloud cover"
ds["cwp_inhomogeneity"].attrs['description'] = "std(cloud_water_path)/mean(cloud_water_path)"
ds["rain_lcf"].attrs["long_name"] = "largest contiguous rainy-cluster fraction"
ds["rain_n_clusters"].attrs["long_name"] = "number of contiguous rainy clusters"

ds['acmcap_cloud_fraction'].attrs["units"] = '1'
ds['acmcap_cloud_fraction'].attrs['long_name'] = "ACM-CAP cloud fraction"
ds['acmcap_cloud_fraction'].attrs['description'] = "ACM-CAP along-track cloud fraction of low clouds (CTH < 3km) in the scene."
ds['cth_all_clouds'].attrs["source_variable"] = 'liquid_classification'
ds["cth_all_clouds"].attrs['long_name'] = "Cloud top height of highest cloud layer"
ds["cth_all_clouds"].attrs['description'] = "In contrast to all other ACM-CAP derived variables in this dataset, which are derived from low-level clouds (CTH < 3km), this variable is derived from all cloud layers, and represents the cloud top height of the highest cloud layer in the scene."
ds["cth_all_clouds"].attrs['units'] = "JSG_height"
ds["ER_passive_perspective"].attrs["long_name"] = "Liquid effective radius computed as (3/2) * LWP / (density_liquid * optical_depth)"
ds['ice_flag'].attrs["long_name"] = 'ice pixel fraction'
ds["ice_flag"].attrs["description"] = "Relative frequency of pixels contaminated by ice clouds (defined as TIR2 < 273 K) in the scene."
ds["multilayer"].attrs["long_name"] = "Relative frequency of pixels with overlying clouds."
ds["multilayer"].attrs["description"] = "Relative frequency of pixels with overlying clouds (defined as cloud base height > 3km) in the scene."
ds["multilayer"].attrs["source_variable"] = "liquid_classification"
ds["N_cloudy_pixels"].attrs["long_name"] = "Mean number of cloudy vertical gates in the scene."
ds["rain_fraction"].attrs["long_name"] = "Relative frequency of rain given a cloud."
ds["rain_fraction"].attrs["description"] = "Relative frequency of profiles with non-zero rain. Non-cloudy profiles are ignored."
ds["rain_fraction"].attrs["intermediate_resolution"] = "1 km"
ds["rain_gini"].attrs["long_name"] = "Gini coefficient of rain rate"
ds["N_cloudy_pixels"].attrs.pop("definition", None)

for name in ["ER", "Nd"]:
    ds[name].attrs["description"] = f"Reduction from ACM-CAP 2D profiles to 1D along-track variable is done by taking the mean over cloudy gates for each vertical profile. The mean of this 1D along-track {name} is then taken over the scene. Non-cloudy profiles are ignored."
for name, label in {"rain_median_volume_diameter": "rain volume diameter", "rain_normalized_number_concentration": "rain normalized number concentration"}.items():
    ds[name].attrs["description"] = f"Reduction from ACM-CAP 2D profiles to 1D along-track {label} is done by taking the mean over rainy gates for each vertical profile. The mean of this 1D along-track {label} is then taken over the scene. Non-rainy profiles are ignored."
for name in ["rain_rate_cbh", "rwc_cbh"]:
    label = ds[name].attrs["long_name"].lower()
    ds[name].attrs["long_name"] = label.capitalize() + " at cloud base"
    ds[name].attrs["description"] = f"The {label} at cloud base height is extracted from the ACM-CAP 2D {ds[name].attrs['source_variable']} profiles. The mean of this 1D along-track {label} at cloud base is then taken over the scene. Non-rainy profiles are ignored."
ds["rain_rate_max"].attrs["long_name"] = "Maximum rain rate"
ds["rain_rate_max"].attrs["description"] = "The maximum rain rate is extracted from the ACM-CAP 2D rain_rate profiles. The mean of this 1D along-track maximum rain rate is then taken over the scene. Non-rainy profiles are ignored."
ds["rain_rate_mean"].attrs["long_name"] = "Mean rain rate"
ds["rain_rate_mean"].attrs["description"] = "The mean rain rate is computed per profile from the ACM-CAP 2D rain_rate variable. The mean of this 1D along-track mean rain rate is then taken over the scene. Non-rainy profiles are ignored."
ds["rain_rate_subcloud"].attrs["long_name"] = "Subcloud rain rate"
ds["rain_rate_subcloud"].attrs["description"] = "The subcloud rain rate is extracted from the ACM-CAP 2D rain_rate profiles by averaging the rain rate below the cloud base height per profile. The mean of this 1D along-track subcloud rain rate is then taken over the scene. Non-rainy profiles are ignored."
ds["rain_rate_surface"].attrs["long_name"] = "Surface rain rate"
ds["rain_rate_surface"].attrs["description"] = "The surface rain rate is defined as the rain rate at JSG_height=235 from the ACM-CAP 2D rain_rate profiles. The mean of this 1D along-track surface rain rate is then taken over the scene. Non-rainy profiles are ignored. Warning: the rain rate is not very reliable close to the surface"
ds["lwp"].attrs["description"] = "Reduction from ACM-CAP 2D profiles to 1D along-track LWP is done by summing over liquid water content for each vertical profile. The mean of this 1D along-track LWP is then taken over the scene. Profiles without low-level clouds are ignored."
ds["rwp"].attrs["description"] = "Reduction from ACM-CAP 2D profiles to 1D along-track RWP is done by summing over rain water content for each vertical profile. The mean of this 1D along-track RWP is then taken over the scene. Non-cloudy profiles are ignored (non-raining but cloudy profiles are taken into account for the average)."
ds["rwp_subcloud"].attrs["description"] = "Reduction from ACM-CAP 2D profiles to 1D along-track subcloud RWP is done by summing over rain water content below the cloud base height for each vertical profile. The mean of this 1D along-track subcloud RWP is then taken over the scene. Non-cloudy profiles are ignored (non-raining but cloudy profiles are taken into account for the average)."


In [20]:
# Inherit base attributes and add statistic names and descriptions.
STATISTICS = {
    "std": ("standard_deviation", "Standard deviation"),
    "median": ("median", "Median"),
    "perc05": ("percentile_05", "5th percentile"),
    "perc25": ("percentile_25", "25th percentile"),
    "perc75": ("percentile_75", "75th percentile"),
    "perc95": ("percentile_95", "95th percentile"),
}

for name in ds:
    base, _, suffix = name.rpartition("_")
    if suffix in STATISTICS and base in ds:
        summary, label = STATISTICS[suffix]
        for key, value in ds[base].attrs.items():
            ds[name].attrs[key] = value
        # Keep the wording for these names.
        if base == "rain_median_volume_diameter" or (suffix == "median" and base in ["ER", "lwp", "Nd", "rwp"]):
            long_name = f"{summary} of {ds[base].attrs['long_name']}"
        else:
            long_name = ds[base].attrs["long_name"].lower()
            if base == "Nd":
                long_name = "liquid number concentration"
            elif base == "N_cloudy_pixels":
                long_name = "number of cloudy vertical gates in the scene."
            long_name = f"{label} of {long_name}"
        ds[name].attrs["long_name"] = long_name
        ds[name].attrs["scene_summary"] = summary
        ds[name].attrs["cell_methods"] = f"scene: {summary}"
        if "description" in ds[base].attrs:
            ds[name].attrs["description"] = ds[base].attrs["description"].replace("The mean of ", f"The {label.lower()} of ").replace("Profiles without low-level clouds", "Non-cloudy profiles")

ds["N_cloudy_pixels_median"].attrs["long_name"] = "Median number of cloudy vertical gates in the scene."


In [21]:
ds.attrs["title"] = "EarthCARE ACM-CAP scene properties joined to MSI neural-network cloud morphology labels"
ds.attrs["summary"] = "EarthCARE ACM-CAP profile variables joined to MSI neural-network scene labels and summarized to 128 km scene scale over four stratocumulus regions."
ds.attrs["processing_level"] = "scene_summary"
ds.attrs["keywords"] = "EarthCARE, ACM-CAP, MSI, stratocumulus, cloud morphology, rain, liquid water path, cloud top height"
ds.attrs["native_acmcap_resolution"] = "1 km along-track profiles"
ds.attrs["scene_observation_unit"] = "128 km MSI neural-network scenes"
ds.attrs["stratocumulus_region_count"] = len(REGION_NAMES)
ds.attrs["stratocumulus_regions"] = ", ".join(REGION_NAMES.values())
ds.attrs["regions"] = str(REGION_BOUNDS)
ds.attrs["license"] = "CC-BY-4.0"
ds.attrs["platform"] = "EarthCARE"
ds.attrs["processing_summary"] = "ACM-CAP 2D profile variables of low-level clouds (CTH < 3km) were reduced to 1D along-track variables, joined to MSI neural-network scene labels, then summarized to 128 km scenes using means, standard deviations, medians, and percentiles."

ds["time_mid"].attrs["source_units"] = ds["time_mid"].attrs.pop("units")

ds


<xarray.Dataset> Size: 49MB
Dimensions:                               (scene: 51861)
Coordinates:
  * scene                                 (scene) int32 207kB 0 1 ... 51860
    scene_id                              (scene) int32 207kB 0 1 ... 51860
    stratus_region_index                  (scene) int32 207kB 0 0 0 0 ... 3 3 3
    time_mid                              (scene) datetime64[ns] 415kB 2024-0...
    lon_mid                               (scene) float64 415kB 6.947 ... -134.6
    lat_mid                               (scene) float64 415kB -3.058 ... 21.94
    orbit                                 (scene) <U6 1MB '01005E' ... '08371E'
    region                                (scene) <U3 622kB 'SEA' ... 'NEP'
Data variables: (12/103)
    along_track_mid                       (scene) float64 415kB 128.5 ... 128.5
    lat_min                               (scene) float64 415kB -3.532 ... 21.47
    lat_max                               (scene) float64 415kB -2.493 ... 22.5
    ice_flag                              (scene) float64 415kB 0.0 0.0 ... 0.0
    cwp_skew                              (scene) float64 415kB 0.3971 ... 7.588
    cwp_inhomogeneity                     (scene) float64 415kB 0.1773 ... 2.263
    ...                                    ...
    rwc_cbh_perc95                        (scene) float64 415kB nan ... 6.485...
    rwc_cbh_perc05                        (scene) float64 415kB nan nan ... 0.0
    rain_gini                             (scene) float64 415kB 0.0 ... 0.9391
    rain_lcf                              (scene) float64 415kB 0.0 ... 0.625
    rain_n_clusters                       (scene) int32 207kB 0 0 0 0 ... 1 2 2
    cat_name                              (scene) <U16 3MB 'Suppressed Cu' .....
Attributes:
    title:                       EarthCARE ACM-CAP scene properties joined to...
    summary:                     EarthCARE ACM-CAP profile variables joined t...
    processing_level:            scene_summary
    keywords:                    EarthCARE, ACM-CAP, MSI, stratocumulus, clou...
    native_acmcap_resolution:    1 km along-track profiles
    scene_observation_unit:      128 km MSI neural-network scenes
    stratocumulus_region_count:  4
    stratocumulus_regions:       SEA, NEA, SEP, NEP
    regions:                     {'SEA': {'lat': (-22.5, -2.5), 'lon': (-5, 1...
    license:                     CC-BY-4.0
    platform:                    EarthCARE
    processing_summary:          ACM-CAP 2D profile variables of low-level cl...

## Export


In [23]:
ds.attrs["date_created"] = datetime.now(timezone.utc).isoformat()
for variable in ds.variables.values():
    variable.encoding["chunks"] = variable.shape

ds.to_zarr(OUTPUT_ZARR, mode="w", consolidated=True, zarr_format=2)

OUTPUT_ZARR


PosixPath('output/earthcare-marine-stratocumulus-transitions/acmcap_judge_stcu/earthcode_acmcap_judge_stcu.zarr')

## Check source values


In [24]:
# Skip the original row index (replaced by scene); check added names against their mappings.
original = read_pandas_fixed_hdf(SOURCE)
original["cat_name"] = original.cats.map(CLASS_NAMES)
original["region"] = original.stratus_region_index.map(REGION_NAMES)
saved = xr.open_zarr(OUTPUT_ZARR, consolidated=True)
pd.testing.assert_frame_equal(
    saved.to_dataframe()[original.columns].reset_index(drop=True),
    original.reset_index(drop=True), check_dtype=False, check_exact=True,
)
